
# 06_Publish_Chunks_To_Supabase

Publica documentos y chunks desde Databricks hacia Supabase/PostgreSQL.

Tareas:

1. Credencial de Supabase desde Databricks Secrets;
2. UPSERT real de documentos;
3. UPSERT real de chunks;
4. Preservación de `chunk_id` cuando el chunk sigue existiendo;
5. Invalidación del embedding cuando cambia el contenido del chunk;
6. Eliminación de chunks obsoletos cuando una nueva ejecución produce menos;
7. Validación por conteo y `content_hash`, no solamente por cantidad;
8. Compatibilidad con el esquema simplificado del script 05;
9. No contiene lógica de Cloud Run: el disparo del job sigue siendo responsabilidad del script 07.


In [0]:

%pip install pg8000


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:

from __future__ import annotations

from datetime import datetime, timezone
import json
import uuid

import pg8000.dbapi

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql import types as T


In [0]:

# ============================================================
# Configuración
# ============================================================

CATALOG_NAME = "workspace"
SCHEMA_NAME = "tfm_pmc"

INVENTORY_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.pmc_inventory"
CHUNK_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.pmc_document_chunks"
PIPELINE_RUNS_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.pipeline_runs"

PIPELINE_NAME = "06_Publish_Chunks_To_Supabase_v3_Clean"

MAX_DOCUMENTS = 20
DOCUMENT_BATCH_SIZE = 100
CHUNK_BATCH_SIZE = 500

RUN_ID = str(uuid.uuid4())
RUN_STARTED_AT = datetime.now(timezone.utc)

print("Run ID:", RUN_ID)


Run ID: c0dfe03d-134a-4de9-a462-905d53136225


In [0]:

# ============================================================
# Supabase / PostgreSQL
# ============================================================

SUPABASE_HOST = "aws-0-ca-central-1.pooler.supabase.com"
SUPABASE_PORT = 6543
SUPABASE_DATABASE = "postgres"
SUPABASE_USER = "postgres.cwymrqsnzgoyzjmhdrkz"

SUPABASE_PASSWORD = dbutils.secrets.get(
    scope="tfm",
    key="supabase-password",
)


def postgres_reader():
    return (
        spark.read
        .format("postgresql")
        .option("host", SUPABASE_HOST)
        .option("port", str(SUPABASE_PORT))
        .option("database", SUPABASE_DATABASE)
        .option("user", SUPABASE_USER)
        .option("password", SUPABASE_PASSWORD)
    )


def get_pg8000_connection():
    return pg8000.dbapi.connect(
        host=SUPABASE_HOST,
        port=SUPABASE_PORT,
        database=SUPABASE_DATABASE,
        user=SUPABASE_USER,
        password=SUPABASE_PASSWORD,
    )


In [0]:

# ============================================================
# Probar conexión y preparar esquema de publicación
# ============================================================

display(
    postgres_reader()
    .option(
        "query",
        '''
        select
            current_database() as database_name,
            current_user as database_user,
            current_timestamp as connection_timestamp
        '''
    )
    .load()
)

connection = get_pg8000_connection()
cursor = connection.cursor()

try:
    cursor.execute(
        "select current_database(), current_user"
    )
    print("pg8000:", cursor.fetchone())

    # Migración idempotente necesaria para validar cambios de contenido.
    cursor.execute(
        '''
        alter table rag.document_chunks
        add column if not exists content_hash text
        '''
    )

    connection.commit()

finally:
    cursor.close()
    connection.close()

print("Supabase schema ready.")


database_name,database_user,connection_timestamp
postgres,postgres,2026-09-13T22:44:11.229Z


pg8000: ['postgres', 'postgres']
Supabase schema ready.


In [0]:

# ============================================================
# Seleccionar documentos pendientes de publicación
# ============================================================

inventory_df = (
    spark.table(INVENTORY_TABLE)
    .filter(F.col("is_latest_version") == True)
    .filter(F.col("chunking_status") == "completed")
    .filter(
        F.coalesce(
            F.col("publication_status"),
            F.lit("pending"),
        ).isin(
            "pending",
            "failed",
        )
    )
    .select(
        "pmcid",
        "article_version",
        "pmid",
        "doi",
        "title",
        "citation",
        "journal",
        "publication_date",
        "publication_year",
        "author_names",
        "pdf_url",
        "xml_url",
        "license_code",
        "corpus_domain",
    )
    .orderBy("pmcid")
    .limit(MAX_DOCUMENTS)
)

documents_selected = inventory_df.count()

print("Documents selected:", documents_selected)

if documents_selected == 0:
    dbutils.notebook.exit(
        "No pending documents to publish."
    )

display(inventory_df)


Documents selected: 20


pmcid,article_version,pmid,doi,title,citation,journal,publication_date,publication_year,author_names,pdf_url,xml_url,license_code,corpus_domain
PMC13538227,PMC13538227.1,42684449,10.1007/s10072-026-09308-6,Hereditary connective tissue disorders in unselected patients with spontaneous cervical artery dissection: a targeted next generation sequencing approach and systematic review,Neurol Sci. 2026 Sep 2;47(9):753. doi: 10.1007/s10072-026-09308-6,Neurological sciences : official journal of the Italian Neurological Society and of the Italian Society of Clinical Neurophysiology,2026 Sep 2,2026,"List(Corradi L, Ferraro C, Tesi F, Abrignani G, Castellini P, Latte L, Trapasso MC, Genovese A, Ritelli MG, Cinquina V, Giliani SC, Magoni M, Menozzi R, Pezzini A)",https://pmc-oa-opendata.s3.amazonaws.com/PMC13538227.1/PMC13538227.1.pdf?md5=96dbbf17b74b615c55f31cca4749cbbe,https://pmc-oa-opendata.s3.amazonaws.com/PMC13538227.1/PMC13538227.1.xml?md5=07b94f2fe91d2f04a955e0bc9f0a5676,CC BY,Rare Genetic Diseases
PMC13538279,PMC13538279.1,42687129,10.1002/acn3.70482,Validation of a Cellular Imaging‐Based Method as a Potential Biomarker for SPG4 Hereditary Spastic Paraplegia,Ann Clin Transl Neurol. 2026 Sep 2:10.1002/acn3.70482. Online ahead of print. doi: 10.1002/acn3.70482,Annals of clinical and translational neurology,2026 Sep 2,2026,"List(Fattorini G, Licursi V, Zanna GD, Dal Canto F, Barghigiani M, Setola N, Rossi S, Funcis A, Santorelli FM, Silvestri G, Casali C, Sardina F, Rinaldo C)",https://pmc-oa-opendata.s3.amazonaws.com/PMC13538279.1/PMC13538279.1.pdf?md5=f14c06178b556380ad81d43bf68ec6bd,https://pmc-oa-opendata.s3.amazonaws.com/PMC13538279.1/PMC13538279.1.xml?md5=05ada6e7f32c1976a3acdd7d23ea5fbe,CC BY,Rare Genetic Diseases
PMC13539412,PMC13539412.1,42694707,10.1530/EO-25-0104,Parathyroid carcinoma: epidemiology and genetics,Endocr Oncol. 2026 Aug 14;6(1):e250104. doi: 10.1530/EO-25-0104,"Endocrine oncology (Bristol, England)",2026 Jan,2026,"List(Betea D, Petrossians P)",https://pmc-oa-opendata.s3.amazonaws.com/PMC13539412.1/PMC13539412.1.pdf?md5=96e5e0f52e1c42e8be83353af60762b8,https://pmc-oa-opendata.s3.amazonaws.com/PMC13539412.1/PMC13539412.1.xml?md5=4af7f054b158ebeda2a40ad8eb693492,CC BY,Rare Genetic Diseases
PMC13540131,PMC13540131.2,42572867,10.1080/17576180.2026.2709280,Development of an electrochemiluminescence-based bridging assay to detect antibodies against a PTH inverse agonist in human plasma,Bioanalysis. Author manuscript; Available in PMC 2026 Sep 5.,Bioanalysis,2026 Jul,2026,"List(Nduwumwami AJ, Wagner EJ, Wang AQ, Fang Y, Tao D, Xu X)",https://pmc-oa-opendata.s3.amazonaws.com/PMC13540131.2/PMC13540131.2.pdf?md5=c1a90cd0c652661c702dc49e08a52106,https://pmc-oa-opendata.s3.amazonaws.com/PMC13540131.2/PMC13540131.2.xml?md5=0acdbdfaa29d5c4fe7759c6bf3663bcf,CC BY,Rare Genetic Diseases
PMC13543812,PMC13543812.1,42648159,10.1016/j.ebiom.2026.106452,"Equity in genome sequencing for rare disease diagnosis: a cross-sectional analysis of data from the UK 100,000 Genomes Project",eBioMedicine. 2026 Aug 26;131:106452. doi: 10.1016/j.ebiom.2026.106452,EBioMedicine,2026 Sep,2026,"List(Tallman S, Moutsianas L, Nguyen T, Cho Y, Mackintosh M, Kasperaviciute D, Brown MA, Ellingford JM, Kuchenbaecker K, Silver MJ)",https://pmc-oa-opendata.s3.amazonaws.com/PMC13543812.1/PMC13543812.1.pdf?md5=6b4e0bb37b7a03d33a4e03f9f37bce1e,https://pmc-oa-opendata.s3.amazonaws.com/PMC13543812.1/PMC13543812.1.xml?md5=684de2fc9f0bad792f1cc7332aecdc77,CC BY,Rare Genetic Diseases
PMC13544149,PMC13544149.1,42699712,10.2147/PHMT.S596065,Identification of a Novel CDH2 Gene Variant in an ACOGS Patient with Concurrent Respiratory Tract Infection: A Case Report,Pediatric Health Med Ther. 2026 Aug 31;17:596065. doi: 10.2147/PHMT.S596065,"Pediatric health, medicine and therapeutics",2026,2026,"List(Lu Y, Fang F, Zhou H, Shu S, Liu X)",https://pmc-oa-opendata.s3.amazonaws.com/PMC13544149.1/PMC13544149.1.pdf?md5=a9897a9c2595c2ea928b9e549785b11e,https://pmc-oa-opendata.s

In [0]:

# ============================================================
# Preparar filas de documentos
# ============================================================

def normalize_publication_date(
    value: str | None,
) -> str | None:
    '''
    Convierte fechas PMC completas a ISO YYYY-MM-DD.

    Las fechas parciales se mantienen como NULL para no inventar
    información que no aparece en la fuente.
    '''

    if not value:
        return None

    value = value.strip()

    formats = [
        "%Y %b %d",
        "%Y %B %d",
    ]

    for date_format in formats:
        try:
            parsed = datetime.strptime(
                value,
                date_format,
            )
            return parsed.date().isoformat()

        except ValueError:
            continue

    return None


document_rows = []
partial_or_unparsed_dates = []

for row in inventory_df.toLocalIterator():

    normalized_date = normalize_publication_date(
        row["publication_date"]
    )

    if (
        row["publication_date"]
        and normalized_date is None
    ):
        partial_or_unparsed_dates.append(
            (
                row["pmcid"],
                row["publication_date"],
            )
        )

    document_rows.append(
        (
            row["pmcid"],
            row["article_version"],
            row["pmid"],
            row["doi"],
            row["title"],
            row["citation"],
            row["journal"],
            normalized_date,
            row["publication_year"],
            json.dumps(
                list(row["author_names"])
                if row["author_names"] is not None
                else [],
                ensure_ascii=False,
            ),
            f"https://pmc.ncbi.nlm.nih.gov/articles/{row['pmcid']}/",
            row["pdf_url"],
            row["xml_url"],
            row["license_code"],
            row["corpus_domain"],
            "PMC",
        )
    )

print(
    "Document rows prepared:",
    len(document_rows),
)

print(
    "Partial/unparsed publication dates set to NULL:",
    len(partial_or_unparsed_dates),
)


Document rows prepared: 20
Partial/unparsed publication dates set to NULL: 14


In [0]:

# ============================================================
# UPSERT de documentos
# ============================================================

DOCUMENT_UPSERT_SQL = '''
INSERT INTO rag.documents (
    pmc_id,
    article_version,
    pmid,
    doi,
    title,
    citation,
    journal,
    publication_date,
    publication_year,
    authors,
    source_url,
    pdf_url,
    xml_url,
    license_code,
    corpus_domain,
    source_system
)
VALUES (
    %s, %s, %s, %s,
    %s, %s, %s, %s,
    %s, %s::jsonb,
    %s, %s, %s,
    %s, %s, %s
)
ON CONFLICT (pmc_id)
DO UPDATE SET
    article_version = EXCLUDED.article_version,
    pmid = EXCLUDED.pmid,
    doi = EXCLUDED.doi,
    title = EXCLUDED.title,
    citation = EXCLUDED.citation,
    journal = EXCLUDED.journal,
    publication_date = EXCLUDED.publication_date,
    publication_year = EXCLUDED.publication_year,
    authors = EXCLUDED.authors,
    source_url = EXCLUDED.source_url,
    pdf_url = EXCLUDED.pdf_url,
    xml_url = EXCLUDED.xml_url,
    license_code = EXCLUDED.license_code,
    corpus_domain = EXCLUDED.corpus_domain,
    source_system = EXCLUDED.source_system
'''

connection = get_pg8000_connection()
cursor = connection.cursor()

try:
    for start in range(
        0,
        len(document_rows),
        DOCUMENT_BATCH_SIZE,
    ):
        batch = document_rows[
            start:start + DOCUMENT_BATCH_SIZE
        ]

        cursor.executemany(
            DOCUMENT_UPSERT_SQL,
            batch,
        )

        connection.commit()

        print(
            f"Documents upserted: "
            f"{start + 1}-{start + len(batch)}"
        )
finally:
    cursor.close()
    connection.close()


Documents upserted: 1-20


In [0]:

# ============================================================
# Recuperar document_id
# ============================================================

document_mapping_df = (
    postgres_reader()
    .option(
        "query",
        '''
        select
            document_id,
            pmc_id,
            article_version
        from rag.documents
        '''
    )
    .load()
)

selected_document_mapping_df = (
    inventory_df
    .select(
        F.col("pmcid").alias("pmc_id"),
        "article_version",
    )
    .join(
        document_mapping_df,
        on=[
            "pmc_id",
            "article_version",
        ],
        how="inner",
    )
)

mapped_documents = (
    selected_document_mapping_df
    .select("pmc_id")
    .distinct()
    .count()
)

print("Documents mapped:", mapped_documents)

if mapped_documents != documents_selected:
    raise RuntimeError(
        "Not all selected documents were mapped in Supabase."
    )


Documents mapped: 20


In [0]:

# ============================================================
# Preparar chunks section-aware
# ============================================================

chunks_source_df = (
    spark.table(CHUNK_TABLE).alias("c")
    .join(
        selected_document_mapping_df.alias("d"),
        on=[
            F.col("c.pmcid") == F.col("d.pmc_id"),
            F.col("c.article_version") == F.col("d.article_version"),
        ],
        how="inner",
    )
    .select(
        F.col("d.document_id").alias("document_id"),
        F.col("d.pmc_id").alias("pmc_id"),

        F.col("c.chunk_index").alias("chunk_index"),
        F.col("c.section_index").alias("section_index"),
        F.col("c.section_chunk_index").alias("section_chunk_index"),

        F.col("c.section_id").alias("section_id"),
        F.col("c.parent_section_id").alias("parent_section_id"),

        F.col("c.section_type").alias("section_type"),
        F.col("c.section_title").alias("section_title"),
        F.col("c.section_path").alias("section_path"),
        F.col("c.section_level").alias("section_level"),

        F.col("c.chunk_text").alias("content"),
        F.col("c.word_count").alias("word_count"),
        F.col("c.character_count").cast("long").alias("character_count"),
        F.col("c.content_hash").alias("content_hash"),
        F.col("c.chunking_method").alias("chunking_method"),
    )
    .orderBy(
        "document_id",
        "chunk_index",
    )
)

expected_chunk_count = chunks_source_df.count()

print("Chunks ready:", expected_chunk_count)

if expected_chunk_count == 0:
    raise RuntimeError(
        "Selected documents have no chunks to publish."
    )

display(chunks_source_df.limit(30))


Chunks ready: 643


document_id,pmc_id,chunk_index,section_index,section_chunk_index,section_id,parent_section_id,section_type,section_title,section_path,section_level,content,word_count,character_count,content_hash,chunking_method
301,PMC13538227,0,0,0,abstract,null,abstract,Abstract,Abstract,0,"Introduction Whether spontaneous cervical artery dissection (sCeAD), the leading cause of ischemic stroke in young adults, represents the manifestation of unrecognized hereditary connective tissue disorders (HCTDs) and whether HCTDs have a major impact in the epidemiology of the disease is a matter of ongoing debate. We aimed at determining the frequency of clinically relevant genetic variants (CRGVs) in a cohort of unselected sCeAD patients by targeted next-generation sequencing (NGS) approach. Methods We designed a high-throughput sequencing panel to identify variants in 38 candidate genes associated with arterial dissection or aneurysm and screened patients with apparently sporadic sCeAD, consecutively referred to one comprehensive stroke center from August 2020 to December 2025. The frequency of known disease-causing and pertinent variants of uncertain significance (VUS) was calculated. Then, we performed a systematic review of all studies evaluating the prevalence of monogenic disorders among sCeAD patients up to December 2025. Results Among 183 patients (males, 51.3%; mean age, 42.0 ± 11.3 years), 2 (1.1%) carried a CeAD-causing variant in COL3A1 ( NM_000090.4:c.2959G > A:p.Gly987Ser) and ABCC6 ( NM_001351800.1:c.3071G > A:p.Arg1024Gln), respectively. In addition, we identified 30 (16.4%) VUS in 25 (13.6%) patients. The analysis of 330 patients across 14 studies yielded a prevalence of carriers of disease-causing variants ranging between 0.4% in series of unselected patients and 23.4% in patients with familial history of",220,1552,35974224f8ab2ccb8e3e9195323ab8c5e074253a0cb2e3859b7418b65430a896,section_word_window_220_overlap_40_v3
301,PMC13538227,1,0,1,abstract,null,abstract,Abstract,Abstract,0,identified 30 (16.4%) VUS in 25 (13.6%) patients. The analysis of 330 patients across 14 studies yielded a prevalence of carriers of disease-causing variants ranging between 0.4% in series of unselected patients and 23.4% in patients with familial history of CeAD. Conclusion Systematic search for rare disease-causing variants should not be recommended in all sCeAD cases but it should be limited to selected individuals with a high pre-test probability to harbor a monogenic disease. Supplementary Information The online version contains supplementary material available at https://doi.org/10.1007/s10072-026-09308-6.,85,619,ec255bf85949fb56f84bdbe31bfdde27975075c7a32748a69192702eb2ce34bf,section_word_window_220_overlap_40_v3
301,PMC13538227,2,1,0,Sec1,null,section,Introduction,Introduction,1,"Arterial dissection is, by definition, the accumulation of blood within the wall of an artery. Once considered uncommon, dissections of carotid or vertebral arteries (cervical artery dissections, CeADs) are now recognized as the major cause of ischemic stroke in young adults, accounting for approximately 20% of cases in patients under 45 years of age. Notwithstanding, the pathogenesis of CeAD is still poorly defined, especially in those cases that occur spontaneously (spontaneous CeAD, sCeAD), without any identifiable precipitating events [ 1 ]. Several arguments point toward a key role played by the connective tissue component of the arterial wall. The finding of composite collagen fibrils and fragmented elastic fibers on electron microscopic examination of skin biopsy specimens in more than half of patients with sCeAD [ 2 – 4 ] and the identification of clinically detectable signs of connective tissue aberration in most sCeAD patients support the hypothesis of a generalized structural connective tissue defect predisposing to disease occurrence, even in sporadic cases without evident signs of a known hereditary connective tissue disorder (HCTD) [ 5, 6 ]. In this view, the 

In [0]:

# ============================================================
# Convertir chunks a filas Python
# ============================================================

chunk_rows = []

for row in chunks_source_df.toLocalIterator():
    chunk_rows.append(
        (
            int(row["document_id"]),
            int(row["chunk_index"]),

            int(row["section_index"]),
            int(row["section_chunk_index"]),

            row["section_id"],
            row["parent_section_id"],

            row["section_type"],
            row["section_title"],
            row["section_path"],

            (
                int(row["section_level"])
                if row["section_level"] is not None
                else None
            ),

            row["content"],
            int(row["word_count"]),
            int(row["character_count"]),
            row["content_hash"],
            row["chunking_method"],
        )
    )

print(
    "Chunk rows prepared:",
    len(chunk_rows),
)


Chunk rows prepared: 643


In [0]:

# ============================================================
# UPSERT de chunks + eliminación de chunks obsoletos
# ============================================================

CHUNK_UPSERT_SQL = '''
INSERT INTO rag.document_chunks (
    document_id,
    chunk_index,

    section_index,
    section_chunk_index,

    section_id,
    parent_section_id,

    section_type,
    section_title,
    section_path,
    section_level,

    content,
    word_count,
    character_count,
    content_hash,
    chunking_method,

    embedding_model,
    embedding_status
)
VALUES (
    %s, %s,
    %s, %s,
    %s, %s,
    %s, %s, %s, %s,
    %s, %s, %s, %s, %s,
    NULL, 'pending'
)
ON CONFLICT (
    document_id,
    chunk_index
)
DO UPDATE SET
    section_index = EXCLUDED.section_index,
    section_chunk_index = EXCLUDED.section_chunk_index,
    section_id = EXCLUDED.section_id,
    parent_section_id = EXCLUDED.parent_section_id,

    section_type = EXCLUDED.section_type,
    section_title = EXCLUDED.section_title,
    section_path = EXCLUDED.section_path,
    section_level = EXCLUDED.section_level,

    content = EXCLUDED.content,
    word_count = EXCLUDED.word_count,
    character_count = EXCLUDED.character_count,
    chunking_method = EXCLUDED.chunking_method,

    embedding =
        CASE
            WHEN rag.document_chunks.content_hash
                 IS DISTINCT FROM EXCLUDED.content_hash
            THEN NULL
            ELSE rag.document_chunks.embedding
        END,

    embedding_model =
        CASE
            WHEN rag.document_chunks.content_hash
                 IS DISTINCT FROM EXCLUDED.content_hash
            THEN NULL
            ELSE rag.document_chunks.embedding_model
        END,

    embedding_status =
        CASE
            WHEN rag.document_chunks.content_hash
                 IS DISTINCT FROM EXCLUDED.content_hash
            THEN 'pending'
            ELSE rag.document_chunks.embedding_status
        END,

    content_hash = EXCLUDED.content_hash
'''

# Como chunk_index es consecutivo por documento (0..N-1),
# cualquier índice >= N es un chunk obsoleto de una versión anterior.
expected_counts = (
    chunks_source_df
    .groupBy(
        "document_id",
    )
    .agg(
        F.count("*").alias(
            "current_chunk_count"
        )
    )
)

expected_count_rows = [
    (
        int(row["document_id"]),
        int(row["current_chunk_count"]),
    )
    for row in expected_counts.toLocalIterator()
]

DELETE_OBSOLETE_SQL = '''
DELETE FROM rag.document_chunks
WHERE document_id = %s
  AND chunk_index >= %s
'''

connection = get_pg8000_connection()
cursor = connection.cursor()

try:
    for start in range(
        0,
        len(chunk_rows),
        CHUNK_BATCH_SIZE,
    ):
        batch = chunk_rows[
            start:start + CHUNK_BATCH_SIZE
        ]

        cursor.executemany(
            CHUNK_UPSERT_SQL,
            batch,
        )

        connection.commit()

        print(
            f"Chunks upserted: "
            f"{start + 1}-{start + len(batch)}"
        )

    cursor.executemany(
        DELETE_OBSOLETE_SQL,
        expected_count_rows,
    )

    connection.commit()

finally:
    cursor.close()
    connection.close()

print("Chunk synchronization completed.")


Chunks upserted: 1-500
Chunks upserted: 501-643
Chunk synchronization completed.


In [0]:

# ============================================================
# Validar conteos y content_hash por documento
# ============================================================

supabase_chunks_df = (
    postgres_reader()
    .option(
        "query",
        '''
        select
            d.pmc_id,
            c.chunk_index,
            c.content_hash
        from rag.documents d
        join rag.document_chunks c
            on c.document_id = d.document_id
        '''
    )
    .load()
)

expected_validation_df = (
    chunks_source_df
    .select(
        "pmc_id",
        "chunk_index",
        F.col("content_hash").alias(
            "databricks_content_hash"
        ),
    )
)

actual_validation_df = (
    supabase_chunks_df
    .select(
        "pmc_id",
        "chunk_index",
        F.col("content_hash").alias(
            "supabase_content_hash"
        ),
    )
)

chunk_validation_df = (
    expected_validation_df.alias("expected")
    .join(
        actual_validation_df.alias("actual"),
        on=[
            "pmc_id",
            "chunk_index",
        ],
        how="left",
    )
    .withColumn(
        "hash_matches",
        F.col("databricks_content_hash")
        == F.col("supabase_content_hash"),
    )
)

validation_df = (
    chunk_validation_df
    .groupBy("pmc_id")
    .agg(
        F.count("*").alias(
            "databricks_chunk_count"
        ),

        F.sum(
            F.when(
                F.col("supabase_content_hash").isNotNull(),
                1,
            ).otherwise(0)
        ).alias(
            "supabase_chunk_count"
        ),

        F.sum(
            F.when(
                F.col("hash_matches") == False,
                1,
            ).otherwise(0)
        ).alias(
            "hash_mismatches"
        ),

        F.sum(
            F.when(
                F.col("supabase_content_hash").isNull(),
                1,
            ).otherwise(0)
        ).alias(
            "missing_chunks"
        ),
    )
    .withColumn(
        "publication_status",
        F.when(
            (F.col("databricks_chunk_count")
             == F.col("supabase_chunk_count"))
            & (F.col("hash_mismatches") == 0)
            & (F.col("missing_chunks") == 0),
            F.lit("completed"),
        ).otherwise(
            F.lit("failed")
        ),
    )
    .withColumn(
        "error_message",
        F.when(
            F.col("publication_status") == "failed",
            F.concat(
                F.lit("Publication validation failed. Expected="),
                F.col("databricks_chunk_count"),
                F.lit(", found="),
                F.col("supabase_chunk_count"),
                F.lit(", missing="),
                F.col("missing_chunks"),
                F.lit(", hash_mismatches="),
                F.col("hash_mismatches"),
            ),
        ),
    )
    .withColumn(
        "updated_at",
        F.current_timestamp(),
    )
)

display(validation_df)

failed_validations = (
    validation_df
    .filter(
        F.col("publication_status") == "failed"
    )
    .count()
)

print(
    "Documents with publication mismatch:",
    failed_validations,
)


pmc_id,databricks_chunk_count,supabase_chunk_count,hash_mismatches,missing_chunks,publication_status,error_message,updated_at
PMC13538279,34,34,0,0,completed,null,2026-09-13T22:45:21.775Z
PMC13543812,38,38,0,0,completed,null,2026-09-13T22:45:21.775Z
PMC13569294,61,61,0,0,completed,null,2026-09-13T22:45:21.775Z
PMC13570588,52,52,0,0,completed,null,2026-09-13T22:45:21.775Z
PMC13540131,35,35,0,0,completed,null,2026-09-13T22:45:21.775Z
PMC13544149,21,21,0,0,completed,null,2026-09-13T22:45:21.775Z
PMC13559883,49,49,0,0,completed,null,2026-09-13T22:45:21.775Z
PMC13552502,48,48,0,0,completed,null,2026-09-13T22:45:21.775Z
PMC13552808,11,11,0,0,completed,null,2026-09-13T22:45:21.775Z
PMC13564258,5,5,0,0,completed,null,2026-09-13T22:45:21.775Z


Documents with publication mismatch: 0


In [0]:

# ============================================================
# Actualizar publication_status
# ============================================================

status_df = (
    validation_df
    .join(
        inventory_df
        .select(
            F.col("pmcid").alias("pmc_id"),
            "article_version",
        ),
        on="pmc_id",
        how="inner",
    )
    .select(
        F.col("pmc_id").alias("pmcid"),
        "article_version",
        "publication_status",
        "error_message",
        "updated_at",
    )
)

(
    DeltaTable.forName(
        spark,
        INVENTORY_TABLE,
    )
    .alias("target")
    .merge(
        status_df.alias("source"),
        '''
        target.pmcid = source.pmcid
        AND target.article_version = source.article_version
        ''',
    )
    .whenMatchedUpdate(
        set={
            "publication_status":
                "source.publication_status",

            "error_message":
                "source.error_message",

            "updated_at":
                "source.updated_at",
        }
    )
    .execute()
)

display(status_df)


pmcid,article_version,publication_status,error_message,updated_at


In [0]:

# ============================================================
# Registrar ejecución en Databricks y Supabase
# ============================================================

RUN_COMPLETED_AT = datetime.now(timezone.utc)

records_processed = status_df.count()

records_failed = (
    status_df
    .filter(
        F.col("publication_status") == "failed"
    )
    .count()
)

records_succeeded = (
    records_processed
    - records_failed
)

run_status = (
    "completed"
    if records_failed == 0
    else "completed_with_errors"
)

databricks_run_row = spark.createDataFrame(
    [(
        RUN_ID,
        PIPELINE_NAME,
        run_status,
        RUN_STARTED_AT,
        RUN_COMPLETED_AT,

        int(MAX_DOCUMENTS),
        int(documents_selected),
        int(records_processed),

        None,  # UPSERT: no se reportan falsos inserts
        None,  # UPSERT: no se separan inserts/updates

        int(records_failed),

        json.dumps({
            "write_driver":
                "pg8000",

            "supabase_port":
                SUPABASE_PORT,

            "expected_chunks":
                expected_chunk_count,

            "document_batch_size":
                DOCUMENT_BATCH_SIZE,

            "chunk_batch_size":
                CHUNK_BATCH_SIZE,

            "document_upsert":
                True,

            "chunk_upsert":
                True,

            "obsolete_chunk_delete":
                True,

            "content_hash_validation":
                True,

            "embedding_invalidation_on_content_change":
                True,
        }),

        None,
    )],
    schema=T.StructType([
        T.StructField("run_id", T.StringType(), False),
        T.StructField("pipeline_name", T.StringType(), False),
        T.StructField("run_status", T.StringType(), False),
        T.StructField("started_at", T.TimestampType(), False),
        T.StructField("completed_at", T.TimestampType(), True),
        T.StructField("records_requested", T.LongType(), True),
        T.StructField("records_found", T.LongType(), True),
        T.StructField("records_processed", T.LongType(), True),
        T.StructField("records_inserted", T.LongType(), True),
        T.StructField("records_updated", T.LongType(), True),
        T.StructField("records_failed", T.LongType(), True),
        T.StructField("execution_metadata", T.StringType(), True),
        T.StructField("error_message", T.StringType(), True),
    ]),
)

databricks_run_row.write.mode(
    "append"
).saveAsTable(
    PIPELINE_RUNS_TABLE
)

SUPABASE_RUN_SQL = '''
INSERT INTO rag.pipeline_runs (
    pipeline_name,
    run_status,
    started_at,
    completed_at,
    records_read,
    records_processed,
    records_succeeded,
    records_failed,
    error_message
)
VALUES (
    %s, %s, %s, %s,
    %s, %s, %s, %s,
    %s
)
'''

connection = get_pg8000_connection()
cursor = connection.cursor()

try:
    cursor.execute(
        SUPABASE_RUN_SQL,
        (
            PIPELINE_NAME,
            run_status,
            RUN_STARTED_AT,
            RUN_COMPLETED_AT,
            int(documents_selected),
            int(records_processed),
            int(records_succeeded),
            int(records_failed),
            None,
        ),
    )

    connection.commit()

finally:
    cursor.close()
    connection.close()

print("Pipeline run registered.")


Pipeline run registered.


In [0]:

# ============================================================
# Auditoría final
# ============================================================

print("===== PUBLICATION STATUS =====")
display(
    status_df
    .orderBy(
        "pmcid",
        "article_version",
    )
)

print("===== SUPABASE COUNTS =====")
display(
    postgres_reader()
    .option(
        "query",
        '''
        select
            (select count(*) from rag.documents)
                as documents,

            (select count(*) from rag.document_chunks)
                as chunks,

            (select count(*) from rag.document_chunks
             where embedding is null)
                as chunks_without_embedding,

            (select count(*) from rag.document_chunks
             where embedding_status = 'pending')
                as chunks_pending_embedding,

            (select count(*) from rag.pipeline_runs)
                as pipeline_runs
        '''
    )
    .load()
)

print("===== PUBLISHED CHUNKS SAMPLE =====")
display(
    postgres_reader()
    .option(
        "query",
        '''
        select
            d.pmc_id,
            d.article_version,
            d.title,
            d.publication_year,

            c.chunk_id,
            c.chunk_index,
            c.section_title,
            c.section_path,
            c.word_count,
            c.content_hash,
            c.embedding_status,

            left(c.content, 300)
                as content_preview

        from rag.document_chunks c

        join rag.documents d
            on d.document_id = c.document_id

        order by
            d.pmc_id,
            c.chunk_index

        limit 30
        '''
    )
    .load()
)

print("===== DATABRICKS PIPELINE RUN =====")
display(
    spark.table(PIPELINE_RUNS_TABLE)
    .filter(
        F.col("run_id") == RUN_ID
    )
)


===== PUBLICATION STATUS =====


pmcid,article_version,publication_status,error_message,updated_at


===== SUPABASE COUNTS =====


documents,chunks,chunks_without_embedding,chunks_pending_embedding,pipeline_runs
320,11137,643,643,2


===== PUBLISHED CHUNKS SAMPLE =====


pmc_id,article_version,title,publication_year,chunk_id,chunk_index,section_title,section_path,word_count,content_hash,embedding_status,content_preview
PMC12289442,PMC12289442.1,POLR3-Related Leukodystrophy: A Qualitative Study on Parents’ Experiences With the Health Care System,2025,1,0,Abstract,Abstract,220,null,completed,"Background: POLR3-related hypomyelinating leukodystrophy (POLR3-HLD) is a rare, inherited neurodegenerative disorder affecting white matter development of the central nervous system. This disorder is characterized by hypomyelination, hypodontia, and hypogonadotropic hypogonadism (4H leukodystrophy)."
PMC12289442,PMC12289442.1,POLR3-Related Leukodystrophy: A Qualitative Study on Parents’ Experiences With the Health Care System,2025,2,1,Abstract,Abstract,70,null,completed,"expressed feeling alone and uncertain, with little guidance provided to them. They also identified perceived gaps in care and challenges faced but found comfort when treated by leukodystrophy experts in specialty clinics. Conclusions: This study will help better inform health care providers, adminis"
PMC12289442,PMC12289442.1,POLR3-Related Leukodystrophy: A Qualitative Study on Parents’ Experiences With the Health Care System,2025,3,2,Introduction,Introduction,220,null,completed,"RNA polymerase III-related hypomyelinating leukodystrophy (POLR3-HLD; MIM: 607694, 614381, 616494, 619310), one of the most common hypomyelinating leukodystrophies, 1 – 6 is an autosomal recessive disorder caused by biallelic pathogenic variants in genes encoding RNA polymerase III (Pol III) subunit"
PMC12289442,PMC12289442.1,POLR3-Related Leukodystrophy: A Qualitative Study on Parents’ Experiences With the Health Care System,2025,4,3,Introduction,Introduction,152,null,completed,"time as communicators and coordinators of their child(ren)’s care and are often required to become experts and advocates due to the scarcity of health care practitioners (HCPs) knowledgeable about POLR3-HLD. 5 Specialized leukodystrophy (LD) centers and clinics offer expert care, information, and su"
PMC12289442,PMC12289442.1,POLR3-Related Leukodystrophy: A Qualitative Study on Parents’ Experiences With the Health Care System,2025,5,4,Study design,Methods > Study design,31,null,completed,"We conducted in-depth, semi-structured interviews with parents of patients with POLR3-HLD. This study was approved by the Research Ethics Board (REB) of the McGill University Health Centre Research Institute (MUHC-RI) (2020–6222)."
PMC12289442,PMC12289442.1,POLR3-Related Leukodystrophy: A Qualitative Study on Parents’ Experiences With the Health Care System,2025,6,5,Participants,Methods > Participants,113,null,completed,"Twenty-four parents (18 mothers and six fathers) were recruited for this study using purposive sampling. 25 – 27 Participants were recruited from the MyeliNeuroGene Biobank (approved by the MUHC-RI REB, 2019–4972). Participants were eligible if they were the parent(s) of a patient with POLR3-HLD, co"
PMC12289442,PMC12289442.1,POLR3-Related Leukodystrophy: A Qualitative Study on Parents’ Experiences With the Health Care System,2025,7,6,Interview guide,Methods > Interview guide,66,null,completed,"A semi-structured interview guide was developed in English by research team members and translated into French ( Supplemental Material ). Interview questions focused on the diagnostic odyssey, the availability and access to care, and the perceived quality of care. It was then pilot-tested with a par"
PMC12289442,PMC12289442.1,POLR3-Related Leukodystrophy: A Qualitative Study on Parents’ Experiences With the Health Care System,2025,8,7,Researchers,Methods > Researchers,76,null,completed,"Our research team was composed of a graduate student (A.L.), a medical student (K.-A.T.), a pediatric neurology resident (P.A.Y.), and an international group of clinician experts in LDs (E.B., F.N., D.P., S.V., S.K., D.R., D.G.M., M.K., D.D.A.P., A.V.), and it was supervised by an expert in qualitat"
PMC12289442,PMC

===== DATABRICKS PIPELINE RUN =====


run_id,pipeline_name,run_status,started_at,completed_at,records_requested,records_found,records_processed,records_inserted,records_updated,records_failed,execution_metadata,error_message
c0dfe03d-134a-4de9-a462-905d53136225,06_Publish_Chunks_To_Supabase_v3_Clean,completed,2026-09-13T22:43:52.808Z,2026-09-13T22:45:40.160Z,20,20,0,null,null,0,"{""write_driver"": ""pg8000"", ""supabase_port"": 6543, ""expected_chunks"": 643, ""document_batch_size"": 100, ""chunk_batch_size"": 500, ""document_upsert"": true, ""chunk_upsert"": true, ""obsolete_chunk_delete"": true, ""content_hash_validation"": true, ""embedding_invalidation_on_content_change"": true}",null
